# Regional Appropriations - Silver

## 1. Transform and load

- Each row represents one year, category, particular, and region.
- Remove `ALL` particulars, `TOTAL` regions, and identical cleaned rows.
- Standardize region codes and names, using codes without dashes.
- Keep Nationwide (`NW`) and Central Office (`CO`) allocations.
- Convert amounts from thousand pesos to pesos; dashes become zero and blanks remain null.
- Keep existing IDs and assign new IDs starting at `1000001`.
- Update changed records and remove missing records from refreshed years.

In [0]:
%sql
-- The job must run with max 1 concurrent run; do not run this by hand while the job is running.

CREATE TABLE IF NOT EXISTS pondolens.dev_silver.regional_appropriations (
  gaa_id INT NOT NULL,
  year INT,
  category STRING,
  particular STRING,
  region_code STRING,
  region_name STRING,
  appropriation_amount DECIMAL(20,2),
  processed_at TIMESTAMP,
  CONSTRAINT regional_appropriations_pk PRIMARY KEY (gaa_id)
)
USING DELTA;

-- Canonical region codes and names.
CREATE OR REPLACE TEMP VIEW regional_gaa_region_map AS
SELECT *
FROM VALUES
  ('NW',    'Nationwide'),
  ('CO',    'Central Office'),
  ('NCR',   'National Capital Region'),
  ('CAR',   'Cordillera Administrative Region'),
  ('I',     'Ilocos Region'),
  ('II',    'Cagayan Valley'),
  ('III',   'Central Luzon'),
  ('IVA',   'CALABARZON'),
  ('IVB',   'MIMAROPA Region'),
  ('V',     'Bicol Region'),
  ('VI',    'Western Visayas'),
  ('NIR',   'Negros Island Region'),
  ('VII',   'Central Visayas'),
  ('VIII',  'Eastern Visayas'),
  ('IX',    'Zamboanga Peninsula'),
  ('X',     'Northern Mindanao'),
  ('XI',    'Davao Region'),
  ('XII',   'SOCCSKSARGEN'),
  ('XIII',  'Caraga'),
  ('BARMM', 'Bangsamoro Autonomous Region in Muslim Mindanao'),
  ('PHL',   'Philippines')
AS m(region_code, region_name);

-- Accept codes, names, REGION-prefixed codes, and combined code/name labels.
CREATE OR REPLACE TEMP VIEW regional_gaa_region_aliases AS
WITH aliases AS (
  SELECT
    region_code,
    region_name,
    EXPLODE(ARRAY(
      region_code,
      region_name,
      CONCAT('REGION ', region_code),
      CONCAT(region_code, ' ', region_name),
      CONCAT('REGION ', region_code, ' ', region_name),
      CONCAT(region_name, ' ', region_code),
      CONCAT(region_name, ' REGION ', region_code)
    )) AS region_alias
  FROM regional_gaa_region_map

  UNION ALL

  SELECT m.region_code, m.region_name, a.region_alias
  FROM VALUES
    ('NW',  'NATIONAWIDE'),
    ('IVB', 'MIMAROPA'),
    ('IVB', 'REGION IVB MIMAROPA'),
    ('IVB', 'IVB MIMAROPA'),
    ('IVB', 'MIMAROPA REGION IVB'),
    ('XIII', 'CARAGA REGION')
  AS a(region_code, region_alias)
  INNER JOIN regional_gaa_region_map AS m
    ON a.region_code = m.region_code
)
SELECT DISTINCT
  REGEXP_REPLACE(UPPER(region_alias), r'[^A-Z0-9]', '') AS region_key,
  region_code,
  region_name
FROM aliases;

-- Exclude exact ALL particulars and TOTAL regions, regardless of letter case.
-- Keep null particulars, as permitted in the original code.
CREATE OR REPLACE TEMP VIEW regional_gaa_bronze_filtered AS
SELECT *
FROM pondolens.dev_bronze.regional_expenditure_gaa
WHERE COALESCE(UPPER(TRIM(particulars)), '') <> 'ALL'
  AND COALESCE(UPPER(TRIM(regions)), '') <> 'TOTAL';

-- Clean source text and remove identical cleaned records.
CREATE OR REPLACE TEMP VIEW regional_gaa_stage AS
SELECT DISTINCT
  CAST(TRIM(CAST(b.year AS STRING)) AS INT) AS year,
  NULLIF(UPPER(TRIM(b.category)), '') AS category,
  NULLIF(UPPER(TRIM(b.particulars)), '') AS particular,
  m.region_code,
  m.region_name,

  -- Dashes become zero; blanks stay null. Convert thousand pesos to pesos.
  CASE
    WHEN TRIM(CAST(b.expenditure_amount AS STRING)) = '-'
      THEN CAST(0 AS DECIMAL(20,2))
    WHEN b.expenditure_amount IS NULL
      OR TRIM(CAST(b.expenditure_amount AS STRING)) = ''
      THEN CAST(NULL AS DECIMAL(20,2))
    ELSE CAST(
      CAST(
        REPLACE(TRIM(CAST(b.expenditure_amount AS STRING)), ',', '')
        AS DECIMAL(20,2)
      ) * 1000
      AS DECIMAL(20,2)
    )
  END AS appropriation_amount
FROM regional_gaa_bronze_filtered AS b
LEFT JOIN regional_gaa_region_aliases AS m
  ON REGEXP_REPLACE(UPPER(TRIM(b.regions)), r'[^A-Z0-9]', '')
     = m.region_key;

-- Bronze rows that cannot be loaded, with their original values and reason.
CREATE OR REPLACE TEMP VIEW regional_gaa_rejected AS
SELECT *
FROM (
  SELECT
    b.source_name,
    b.source_page,
    b.year,
    b.category,
    b.particulars,
    b.regions,
    b.expenditure_amount,
    b.unit_amount,
    CONCAT_WS('; ',
      CASE
        WHEN TRY_CAST(TRIM(CAST(b.year AS STRING)) AS INT) IS NULL
        THEN 'invalid year'
      END,
      CASE
        WHEN NULLIF(TRIM(b.category), '') IS NULL
        THEN 'missing category'
      END,
      CASE
        WHEN NULLIF(
          REGEXP_REPLACE(UPPER(TRIM(b.regions)), r'[^A-Z0-9]', ''),
          ''
        ) IS NULL
        THEN 'missing region'
        WHEN m.region_code IS NULL
        THEN 'unmapped region'
      END,
      CASE
        WHEN TRIM(CAST(b.expenditure_amount AS STRING)) NOT IN ('', '-')
          AND TRY_CAST(
            REPLACE(TRIM(CAST(b.expenditure_amount AS STRING)), ',', '')
            AS DECIMAL(20,2)
          ) IS NULL
        THEN 'invalid amount'
      END,
      CASE
        WHEN b.unit_amount IS NULL
          OR UPPER(TRIM(b.unit_amount)) NOT LIKE '%THOUSAND%'
        THEN 'unit is not thousand pesos'
      END
    ) AS rejection_reason
  FROM regional_gaa_bronze_filtered AS b
  LEFT JOIN regional_gaa_region_aliases AS m
    ON REGEXP_REPLACE(UPPER(TRIM(b.regions)), r'[^A-Z0-9]', '')
       = m.region_key
)
WHERE rejection_reason <> '';

-- To inspect rejected rows: SELECT * FROM regional_gaa_rejected.
SELECT ASSERT_TRUE(
  COUNT(*) = 0,
  'Rejected Bronze rows found (bad year, key, region, amount or unit). See regional_gaa_rejected.'
)
FROM regional_gaa_rejected;

-- Particular may be null; comparisons below remain null-safe.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(year)
  AND COUNT(*) = COUNT(category)
  AND COUNT(*) = COUNT(region_code)
  AND COUNT(*) = COUNT(region_name)
  AND COUNT(*) = COUNT(
    DISTINCT STRUCT(year, category, particular, region_code)
  ),
  'Missing key fields or conflicting source records.'
)
FROM regional_gaa_stage;

-- Normalize existing keys for matching while retaining their IDs.
-- Mark existing ALL/TOTAL rows for deletion.
CREATE OR REPLACE TEMP VIEW regional_gaa_target_clean AS
SELECT
  t.gaa_id,
  t.year,
  NULLIF(UPPER(TRIM(t.category)), '') AS category,
  NULLIF(UPPER(TRIM(t.particular)), '') AS particular,
  m.region_code,
  m.region_name,
  t.appropriation_amount,
  (
    COALESCE(UPPER(TRIM(t.particular)), '') = 'ALL'
    OR COALESCE(UPPER(TRIM(t.region_code)), '') = 'TOTAL'
  ) AS excluded
FROM pondolens.dev_silver.regional_appropriations AS t
LEFT JOIN regional_gaa_region_aliases AS m
  ON REGEXP_REPLACE(UPPER(TRIM(t.region_code)), r'[^A-Z0-9]', '')
     = m.region_key;

-- Check all existing IDs before merging.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT gaa_id)
  AND COUNT_IF(gaa_id <= 1000000) = 0,
  'Missing or duplicate IDs, or IDs at or below 1000000.'
)
FROM regional_gaa_target_clean;

-- Check retained records after region normalization.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(year)
  AND COUNT(*) = COUNT(category)
  AND COUNT(*) = COUNT(region_code)
  AND COUNT(*) = COUNT(region_name)
  AND COUNT(*) = COUNT(
    DISTINCT STRUCT(year, category, particular, region_code)
  ),
  'Missing keys, unmapped existing regions or duplicate normalized records.'
)
FROM regional_gaa_target_clean
WHERE NOT excluded;

-- Keep the 80% safeguard, excluding intentionally removed ALL/TOTAL rows.
SELECT ASSERT_TRUE(
  COUNT(*) = 0,
  'A Bronze year has under 80% of the Silver rows; possible partial load.'
)
FROM (
  SELECT t.year
  FROM regional_gaa_target_clean AS t
  INNER JOIN (
    SELECT year, COUNT(*) AS n
    FROM regional_gaa_stage
    GROUP BY year
  ) AS s ON t.year = s.year
  WHERE NOT t.excluded
  GROUP BY t.year, s.n
  HAVING s.n < 0.8 * COUNT(*)
);

CREATE OR REPLACE TEMP VIEW regional_gaa_numbered AS
WITH new_records AS (
  -- Assign IDs only to records not already in Silver.
  SELECT s.*
  FROM regional_gaa_stage AS s
  LEFT ANTI JOIN regional_gaa_target_clean AS t
    ON NOT t.excluded
    AND s.year = t.year
    AND s.category = t.category
    AND s.particular <=> t.particular
    AND s.region_code = t.region_code
),
last_id AS (
  -- Empty table starts at 1000001; otherwise continue from the highest ID.
  SELECT
    COALESCE(MAX(CAST(gaa_id AS BIGINT)), 1000000) AS last_number
  FROM pondolens.dev_silver.regional_appropriations
)
SELECT
  s.*,
  l.last_number
    + ROW_NUMBER() OVER (
        ORDER BY s.year, s.category, s.particular, s.region_code
      ) AS next_id
FROM new_records AS s
CROSS JOIN last_id AS l;

SELECT ASSERT_TRUE(
  COUNT(*) = 0,
  'New IDs must be between 1000001 and 2147483647.'
)
FROM regional_gaa_numbered
WHERE next_id NOT BETWEEN 1000001 AND 2147483647;

CREATE OR REPLACE TEMP VIEW regional_gaa_merge_source AS
-- Existing records retain their IDs.
SELECT
  t.gaa_id,
  s.year,
  s.category,
  s.particular,
  s.region_code,
  s.region_name,
  s.appropriation_amount,
  FALSE AS removed
FROM regional_gaa_stage AS s
INNER JOIN regional_gaa_target_clean AS t
  ON NOT t.excluded
  AND s.year = t.year
  AND s.category = t.category
  AND s.particular <=> t.particular
  AND s.region_code = t.region_code

UNION ALL

SELECT
  CAST(next_id AS INT) AS gaa_id,
  year,
  category,
  particular,
  region_code,
  region_name,
  appropriation_amount,
  FALSE AS removed
FROM regional_gaa_numbered

UNION ALL

-- Normalize retained Silver years absent from this Bronze batch.
SELECT
  t.gaa_id,
  t.year,
  t.category,
  t.particular,
  t.region_code,
  t.region_name,
  t.appropriation_amount,
  FALSE AS removed
FROM regional_gaa_target_clean AS t
LEFT ANTI JOIN (
  SELECT DISTINCT year FROM regional_gaa_stage
) AS y ON t.year = y.year
WHERE NOT t.excluded

UNION ALL

-- Silver records missing from a refreshed Bronze year are deleted.
SELECT
  t.gaa_id,
  t.year,
  t.category,
  t.particular,
  t.region_code,
  t.region_name,
  CAST(NULL AS DECIMAL(20,2)) AS appropriation_amount,
  TRUE AS removed
FROM regional_gaa_target_clean AS t
LEFT SEMI JOIN (
  SELECT DISTINCT year FROM regional_gaa_stage
) AS y ON t.year = y.year
LEFT ANTI JOIN regional_gaa_stage AS s
  ON t.year = s.year
  AND t.category = s.category
  AND t.particular <=> s.particular
  AND t.region_code = s.region_code
WHERE NOT t.excluded

UNION ALL

-- Remove existing ALL/TOTAL rows, including years absent from the batch.
SELECT
  gaa_id,
  year,
  category,
  particular,
  region_code,
  region_name,
  CAST(NULL AS DECIMAL(20,2)) AS appropriation_amount,
  TRUE AS removed
FROM regional_gaa_target_clean
WHERE excluded;

MERGE INTO pondolens.dev_silver.regional_appropriations AS t
USING regional_gaa_merge_source AS s
ON t.gaa_id = s.gaa_id

WHEN MATCHED AND s.removed THEN DELETE

-- Update changed amounts or text; retain the existing ID.
WHEN MATCHED AND NOT s.removed AND (
  NOT (t.category <=> s.category)
  OR NOT (t.particular <=> s.particular)
  OR NOT (t.region_code <=> s.region_code)
  OR NOT (t.region_name <=> s.region_name)
  OR NOT (t.appropriation_amount <=> s.appropriation_amount)
)
THEN UPDATE SET
  t.category = s.category,
  t.particular = s.particular,
  t.region_code = s.region_code,
  t.region_name = s.region_name,
  t.appropriation_amount = s.appropriation_amount,
  t.processed_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED AND NOT s.removed THEN INSERT (
  gaa_id,
  year,
  category,
  particular,
  region_code,
  region_name,
  appropriation_amount,
  processed_at
)
VALUES (
  s.gaa_id,
  s.year,
  s.category,
  s.particular,
  s.region_code,
  s.region_name,
  s.appropriation_amount,
  CURRENT_TIMESTAMP()
);

## 2. Data quality checks

- Run after Section 1 succeeds
- **PASS:** No issues found for this check.
- **FAIL:** Invalid source rows, keys, text, region mappings, timestamps, excluded rows, or reconciliation differences.
- **REVIEW:** Empty source, null amounts, or negative amounts requiring inspection.

In [0]:
%sql
-- Read-only report. Run after the transformation succeeds; no temporary views required.
WITH region_map AS (
  SELECT *
  FROM VALUES
    ('NW',    'Nationwide'),
    ('CO',    'Central Office'),
    ('NCR',   'National Capital Region'),
    ('CAR',   'Cordillera Administrative Region'),
    ('I',     'Ilocos Region'),
    ('II',    'Cagayan Valley'),
    ('III',   'Central Luzon'),
    ('IVA',   'CALABARZON'),
    ('IVB',   'MIMAROPA Region'),
    ('V',     'Bicol Region'),
    ('VI',    'Western Visayas'),
    ('NIR',   'Negros Island Region'),
    ('VII',   'Central Visayas'),
    ('VIII',  'Eastern Visayas'),
    ('IX',    'Zamboanga Peninsula'),
    ('X',     'Northern Mindanao'),
    ('XI',    'Davao Region'),
    ('XII',   'SOCCSKSARGEN'),
    ('XIII',  'Caraga'),
    ('BARMM', 'Bangsamoro Autonomous Region in Muslim Mindanao'),
    ('PHL',   'Philippines')
  AS m(region_code, region_name)
),
alias_labels AS (
  SELECT
    region_code,
    region_name,
    EXPLODE(ARRAY(
      region_code,
      region_name,
      CONCAT('REGION ', region_code),
      CONCAT(region_code, ' ', region_name),
      CONCAT('REGION ', region_code, ' ', region_name),
      CONCAT(region_name, ' ', region_code),
      CONCAT(region_name, ' REGION ', region_code)
    )) AS region_alias
  FROM region_map

  UNION ALL

  SELECT m.region_code, m.region_name, a.region_alias
  FROM VALUES
    ('NW',   'NATIONAWIDE'),
    ('IVB',  'MIMAROPA'),
    ('IVB',  'REGION IVB MIMAROPA'),
    ('IVB',  'IVB MIMAROPA'),
    ('IVB',  'MIMAROPA REGION IVB'),
    ('XIII', 'CARAGA REGION')
  AS a(region_code, region_alias)
  INNER JOIN region_map AS m
    ON a.region_code = m.region_code
),
region_aliases AS (
  SELECT DISTINCT
    REGEXP_REPLACE(UPPER(region_alias), r'[^A-Z0-9]', '') AS region_key,
    region_code,
    region_name
  FROM alias_labels
),
bronze AS (
  SELECT *
  FROM pondolens.dev_bronze.regional_expenditure_gaa
),
bronze_filtered AS (
  -- Exclude exact ALL particulars and TOTAL regions, regardless of letter case.
  SELECT *
  FROM bronze
  WHERE COALESCE(UPPER(TRIM(particulars)), '') <> 'ALL'
    AND COALESCE(UPPER(TRIM(regions)), '') <> 'TOTAL'
),
bronze_prepared AS (
  SELECT
    TRY_CAST(TRIM(CAST(b.year AS STRING)) AS INT) AS year,
    NULLIF(UPPER(TRIM(b.category)), '') AS category,
    NULLIF(UPPER(TRIM(b.particulars)), '') AS particular,
    m.region_code,
    m.region_name,
    TRIM(CAST(b.expenditure_amount AS STRING)) AS amount_text,
    TRY_CAST(
      REPLACE(TRIM(CAST(b.expenditure_amount AS STRING)), ',', '')
      AS DECIMAL(20,2)
    ) AS parsed_amount,
    b.unit_amount
  FROM bronze_filtered AS b
  LEFT JOIN region_aliases AS m
    ON REGEXP_REPLACE(UPPER(TRIM(b.regions)), r'[^A-Z0-9]', '')
       = m.region_key
),
bronze_cleaned AS (
  -- Match Silver treatment: dashes become zero; thousand pesos become pesos.
  SELECT
    *,
    CASE
      WHEN amount_text = '-' THEN CAST(0 AS DECIMAL(20,2))
      WHEN amount_text IS NULL OR amount_text = ''
        THEN CAST(NULL AS DECIMAL(20,2))
      ELSE TRY_CAST(parsed_amount * 1000 AS DECIMAL(20,2))
    END AS appropriation_amount
  FROM bronze_prepared
),
rejected AS (
  SELECT *
  FROM bronze_cleaned
  WHERE year IS NULL
     OR category IS NULL
     OR region_code IS NULL
     OR (
       amount_text NOT IN ('', '-')
       AND appropriation_amount IS NULL
     )
     OR unit_amount IS NULL
     OR UPPER(TRIM(unit_amount)) NOT LIKE '%THOUSAND%'
),
source_stage AS (
  SELECT DISTINCT
    year,
    category,
    particular,
    region_code,
    region_name,
    appropriation_amount
  FROM bronze_cleaned
),
silver AS (
  SELECT *
  FROM pondolens.dev_silver.regional_appropriations
),
refreshed AS (
  SELECT t.*
  FROM silver AS t
  LEFT SEMI JOIN (
    SELECT DISTINCT year FROM source_stage
  ) AS y ON t.year = y.year
),
source_minus_target AS (
  SELECT
    year, category, particular,
    region_code, region_name, appropriation_amount
  FROM source_stage

  EXCEPT ALL

  SELECT
    year, category, particular,
    region_code, region_name, appropriation_amount
  FROM refreshed
),
target_minus_source AS (
  SELECT
    year, category, particular,
    region_code, region_name, appropriation_amount
  FROM refreshed

  EXCEPT ALL

  SELECT
    year, category, particular,
    region_code, region_name, appropriation_amount
  FROM source_stage
),
checks AS (
  -- Totals are informational; remaining counts represent issues.
  SELECT
    -1 AS sort_order,
    'Total Bronze rows (before filtering)' AS check_name,
    COUNT(*) AS row_count,
    'INFO' AS issue_status
  FROM bronze

  UNION ALL

  SELECT 0, 'Total Silver rows',
    COUNT(*), 'INFO'
  FROM silver

  UNION ALL

  SELECT 1, 'Rejected Bronze rows',
    COUNT(*), 'FAIL'
  FROM rejected

  UNION ALL

  SELECT 2, 'Empty source stage',
    IF(COUNT(*) = 0, 1, 0), 'REVIEW'
  FROM source_stage

  UNION ALL

  SELECT 3, 'Missing required Silver keys or region names',
    COUNT(*), 'FAIL'
  FROM silver
  WHERE gaa_id IS NULL
     OR year IS NULL
     OR NULLIF(TRIM(category), '') IS NULL
     OR NULLIF(TRIM(region_code), '') IS NULL
     OR NULLIF(TRIM(region_name), '') IS NULL

  UNION ALL

  SELECT 4, 'Duplicate Silver IDs (extra rows)',
    COUNT(*) - COUNT(DISTINCT gaa_id), 'FAIL'
  FROM silver
  WHERE gaa_id IS NOT NULL

  UNION ALL

  SELECT 5, 'Duplicate Silver composite keys (extra rows)',
    COUNT(*) - COUNT(
      DISTINCT STRUCT(year, category, particular, region_code)
    ), 'FAIL'
  FROM silver

  UNION ALL

  SELECT 6, 'IDs outside allowed range',
    COUNT(*), 'FAIL'
  FROM silver
  WHERE gaa_id NOT BETWEEN 1000001 AND 2147483647

  UNION ALL

  SELECT 7, 'Unstandardized Silver text',
    COUNT(*), 'FAIL'
  FROM silver
  WHERE NOT (category <=> NULLIF(UPPER(TRIM(category)), ''))
     OR NOT (particular <=> NULLIF(UPPER(TRIM(particular)), ''))
     OR NOT (region_code <=> NULLIF(UPPER(TRIM(region_code)), ''))
     OR NOT (region_name <=> NULLIF(TRIM(region_name), ''))
     OR INSTR(region_code, '-') > 0

  UNION ALL

  SELECT 8, 'Missing processing timestamps',
    COUNT(*), 'FAIL'
  FROM silver
  WHERE processed_at IS NULL

  UNION ALL

  SELECT 9, 'Source rows missing or different in Silver',
    COUNT(*), 'FAIL'
  FROM source_minus_target

  UNION ALL

  SELECT 10, 'Extra or different Silver rows in refreshed years',
    COUNT(*), 'FAIL'
  FROM target_minus_source

  UNION ALL

  SELECT 11, 'Null amounts requiring source review',
    COUNT(*), 'REVIEW'
  FROM silver
  WHERE appropriation_amount IS NULL

  UNION ALL

  SELECT 12, 'Negative amounts requiring source review',
    COUNT(*), 'REVIEW'
  FROM silver
  WHERE appropriation_amount < 0

  UNION ALL

  SELECT 13, 'Invalid region code or name mapping',
    COUNT(*), 'FAIL'
  FROM silver AS s
  LEFT ANTI JOIN region_map AS m
    ON s.region_code = m.region_code
    AND s.region_name = m.region_name

  UNION ALL

  SELECT 14, 'Excluded ALL particulars or TOTAL regions retained',
    COUNT(*), 'FAIL'
  FROM silver
  WHERE UPPER(TRIM(particular)) = 'ALL'
     OR UPPER(TRIM(region_code)) = 'TOTAL'
     OR UPPER(TRIM(region_name)) = 'TOTAL'
)
SELECT
  check_name,
  row_count,
  CASE
    WHEN issue_status = 'INFO' THEN 'INFO'
    WHEN row_count = 0 THEN 'PASS'
    ELSE issue_status
  END AS status
FROM checks
ORDER BY sort_order;

## 3. DQ results

- Bronze: **4,780 rows**
- Silver: **4,256 rows**
- Silver matches the cleaned source for refreshed years.
- No duplicate keys, invalid region mappings, or remaining `ALL`/`TOTAL` rows were found.
- **126 rows have null amounts** 
- No negative amounts were found.